# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://aerc.assam.gov.in/")
links

['#skipCont',
 'javascript:void(0);',
 'javascript:void(0);',
 'javascript:void(0);',
 'javascript:void(0);',
 'javascript:void(0);',
 'http://assam.gov.in',
 'http://power.assam.gov.in/',
 'http://aerc.assam.gov.in/',
 'http://power.assam.gov.in',
 '/',
 '/about-us',
 '/about-us/about-aerc',
 '/about-us/what-we-do',
 '/about-us/who-we-are',
 '/information-services',
 '/information-services/orders',
 '/information-services/petitions',
 '/information-services/annual-reports',
 '/information-services/renewable-purchase-obligation-rpo',
 '/information-services/budget',
 '/information-services/important-links',
 '/documents/acts-17',
 '/documents/acts-17',
 '/documents/rules-policies',
 '/documents/other-meetings-record-notes',
 '/documents/regulations',
 'https://aerc.assam.gov.in/portlet-innerpage/consolidated-regulations',
 'https://aerc.assam.gov.in/portlet-innerpage/individual-regulations',
 'https://aerc.assam.gov.in/portlet-innerpage/repealed-regulations',
 'https://aerc.assam.gov.i

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://aerc.assam.gov.in/"))


Here is the list of links on the website https://aerc.assam.gov.in/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#skipCont
javascript:void(0);
javascript:void(0);
javascript:void(0);
javascript:void(0);
javascript:void(0);
http://assam.gov.in
http://power.assam.gov.in/
http://aerc.assam.gov.in/
http://power.assam.gov.in
/
/about-us
/about-us/about-aerc
/about-us/what-we-do
/about-us/who-we-are
/information-services
/information-services/orders
/information-services/petitions
/information-services/annual-reports
/information-services/renewable-purchase-obligation-rpo
/information-services/budget
/information-services/important-links
/documents/acts-17
/documents/acts-17
/documents/rules-policies
/documents/other-meetings-record-notes
/documents/regulations
https://aerc.assam.gov.in/portlet-innerpage/co

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://aerc.assam.gov.in/")

{'links': [{'type': 'home page', 'url': 'https://aerc.assam.gov.in/'},
  {'type': 'about page',
   'url': 'https://aerc.assam.gov.in/about-us/about-aerc'},
  {'type': 'about page',
   'url': 'https://aerc.assam.gov.in/about-us/what-we-do'},
  {'type': 'about page',
   'url': 'https://aerc.assam.gov.in/about-us/who-we-are'},
  {'type': 'contact page', 'url': 'https://aerc.assam.gov.in/contact-us'},
  {'type': 'careers page',
   'url': 'https://aerc.assam.gov.in/resource/careers'},
  {'type': 'annual reports page',
   'url': 'https://aerc.assam.gov.in/information-services/annual-reports'},
  {'type': 'budget page',
   'url': 'https://aerc.assam.gov.in/information-services/budget'},
  {'type': 'renewable purchase obligation (RPO) page',
   'url': 'https://aerc.assam.gov.in/information-services/renewable-purchase-obligation-rpo'},
  {'type': 'schemes page', 'url': 'https://aerc.assam.gov.in/schemes'},
  {'type': 'faq page', 'url': 'https://aerc.assam.gov.in/faq'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://aerc.assam.gov.in/"))

## Landing Page:

Home | Assam Electricity Regulatory Commission (AERC) | Government Of Assam, India

Skip to content
A
+
A
A
-
A
A
I am now accessing as a :
Government Of Assam
Power
Assam Electricity Regulatory Commission (AERC)
Main
Home
About Us
About AERC
What We Do
Who We Are
You can find information on Our Ministers, Key Officials, Our Vision,Mission and Functions and more details about our department here.
Information & Services
Orders
Petitions
Annual Reports
Renewable Purchase Obligation (RPO)
Budget
Important Links
We have tried to link all Information & Services together to help you locate them faster.
Documents
Acts
Rules & Policies
Other Meetings Record Notes
A document repository where all types of the documents of the organization can be searched and located in the shortest possible time.
Regulations
Consolidated Regulations
Individual Regulations
Repealed Regulations
Draft Regulations
Schemes
Deendayal Upadhyaya Gram Jyoti Yojana
Uday Ujjawal Scheme
Integrated Power De

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("AERC", "https://aerc.assam.gov.in/")

'\nYou are looking at a company called: AERC\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome | Assam Electricity Regulatory Commission (AERC) | Government Of Assam, India\n\nSkip to content\nA\n+\nA\nA\n-\nA\nA\nI am now accessing as a :\nGovernment Of Assam\nPower\nAssam Electricity Regulatory Commission (AERC)\nMain\nHome\nAbout Us\nAbout AERC\nWhat We Do\nWho We Are\nYou can find information on Our Ministers, Key Officials, Our Vision,Mission and Functions and more details about our department here.\nInformation & Services\nOrders\nPetitions\nAnnual Reports\nRenewable Purchase Obligation (RPO)\nBudget\nImportant Links\nWe have tried to link all Information & Services together to help you locate them faster.\nDocuments\nActs\nRules & Policies\nOther Meetings Record Notes\nA document repository where all types of the documents of the organi

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("AERC", "https://aerc.assam.gov.in/")

# Assam Electricity Regulatory Commission (AERC)  
*Government of Assam, India*

---

## Overview  
The Assam Electricity Regulatory Commission (AERC) is the authoritative government body overseeing electricity regulation within the state of Assam. AERC ensures a fair, efficient, and transparent electricity market, balancing the interests of consumers, utilities, and government policies. As a regulatory commission, it plays a critical role in shaping Assam's power sector through policy formulation, implementation, and compliance monitoring.

---

## Vision & Mission  
- **Vision:** To promote sustainable, reliable, and affordable electricity supply in Assam fostering economic growth and social welfare.  
- **Mission:** To regulate the electricity sector impartially by ensuring transparency, promoting competition, protecting consumer rights, and encouraging renewable energy adoption.

---

## What We Do  
AERC’s mandate encompasses a broad array of regulatory functions including:  
- Regulating tariffs for electricity generation, transmission, and distribution to balance affordability and sector sustainability.  
- Monitoring and enforcing compliance with electricity laws, rules, and policies enacted by the Government of Assam and India.  
- Overseeing implementation of key schemes such as:  
  - **Deendayal Upadhyaya Gram Jyoti Yojana (DDUGJY)**  
  - **Uday Ujjawal Scheme**  
  - **Integrated Power Development Scheme (IPDS)**  
  - **Perform, Achieve, Trade (PAT) Scheme**  
- Managing petitions and complaints from consumers ensuring timely resolution and promoting consumer awareness.  
- Ensuring adherence to the Renewable Purchase Obligation (RPO) to boost clean energy integration in Assam’s power mix.  
- Publishing annual reports, budget details, and other regulatory orders transparently for public access.

---

## Customers & Stakeholders  
- Residential and industrial electricity consumers across Assam.  
- Power generation companies, transmission, and distribution utilities operating within the state.  
- Government departments and policymakers.  
- Renewable energy developers and investors.  
- General public seeking electricity sector information or grievance redressal.

---

## Company Culture  
AERC functions as a government regulatory body driven by values of:  
- **Transparency:** Committed to open access to information including orders, regulations, and budget details.  
- **Accountability:** Ensuring regulatory actions serve the public interest and adhere to statutory mandates.  
- **Service Excellence:** Focusing on prompt resolution of consumer grievances and facilitating ease of doing business in the power sector.  
- **Innovation:** Supporting policies and schemes that promote renewable energy, efficiency improvements, and technological advancements in power.  
- **Collaboration:** Working closely with all stakeholders to deliver equitable and sustainable electricity regulation.

---

## Careers at AERC  
AERC offers career opportunities for professionals passionate about the power sector and public service. Potential roles include regulatory officers, legal experts, technical analysts, policy advisors, and administrative staff. Working at AERC provides a unique chance to contribute to Assam’s energy sustainability and development.

- Visit the **Careers** section on the AERC website for current openings and recruitment notifications.  
- Join a dynamic team dedicated to shaping the future of Assam’s electricity landscape.

---

## Contact & Resources  
- **Website:** Assam Electricity Regulatory Commission (AERC) hosted under the Government of Assam portal.  
- Extensive repository of documents including orders, policies, budget reports, schemes, and meeting records are available online.  
- User-friendly support including FAQs, complaint filing mechanisms, feedback forms, and direct contact resources.

---

Assam Electricity Regulatory Commission (AERC) is your trusted partner for a sustainable, transparent, and consumer-friendly electricity sector in Assam. Join us in empowering every citizen with reliable and affordable power while advancing towards a greener future.

---

*Last updated: October 2024*  
*Website maintained by National Informatics Centre (NIC), Government of Assam.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [17]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("AERC", "https://aerc.assam.gov.in/")

# Assam Electricity Regulatory Commission (AERC)  
*Government of Assam, India*

---

## Overview

The Assam Electricity Regulatory Commission (AERC) is a governmental authority committed to regulating and overseeing the electricity sector in Assam. It plays a pivotal role in enhancing the power sector's efficiency, promoting sound policies, and ensuring consumer-friendly electricity supply aligned with the Government of Assam’s vision for sustainable development.

---

## Vision & Mission

AERC aims to ensure the availability of reliable, affordable, and quality electricity to all consumers in Assam. Its mission includes:

- Regulating electricity tariffs fairly and transparently.
- Promoting competition and efficiency in the power sector.
- Supporting renewable energy initiatives and sustainable power development.
- Protecting consumer interests through grievance redressal and awareness.

---

## Functions and Services

- **Regulation** of tariffs and licensing in the power sector.
- Monitoring **Renewable Purchase Obligations (RPO)** to promote green energy.
- Handling **Petitions and Complaints** related to electricity supply.
- Issuance of **Orders, Notifications, and Annual Reports** for transparency.
- Overseeing implementation of key government schemes like:
  - Deendayal Upadhyaya Gram Jyoti Yojana
  - Uday Ujjawal Scheme
  - Integrated Power Development Scheme (IPDS)
  - Perform Achieve Trade (PAT) Scheme

---

## Customers & Stakeholders

AERC serves a broad range of stakeholders, including:

- Consumers of electricity across Assam — residential, commercial, and industrial.
- Electricity distribution companies, generators, and transmission licensees.
- Government ministries and departments.
- Renewable energy producers and investors.
- Policy makers and regulators.

The commission ensures the interests of these diverse stakeholders are balanced, facilitating the sustainable growth of Assam’s power sector.

---

## Company Culture

As a government regulatory body, AERC embodies principles of transparency, accountability, service orientation, and innovation. It fosters a collaborative environment among its officials and stakeholders, driven by a commitment to public welfare and sustainable energy development.

---

## Careers at AERC

AERC offers career opportunities for professionals interested in regulation, policy analysis, energy management, and administrative roles within the government sector. The organization values integrity, expertise, and dedication to public service.

---

## Contact & More Information

- For policies, orders, petitions, and schemes, visit the official website of AERC under the Department of Power, Government of Assam.
- Engage with consumer awareness programs and file complaints through designated portals.
- Access resource documents including Acts, Rules, Regulations, and Annual Reports easily through the website.

---

*Designed and maintained in collaboration with the National Informatics Centre (NIC), Government of Assam.*

---

**Empowering Assam through transparent and sustainable electricity regulation.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>